In [3]:
# Empirical Law 4.1 step-size check with MOSEK screening and SDPA confirmation of candidate improvements.
import numpy as np
from pathlib import Path
import sys

for path in (Path.cwd(), Path.cwd() / "certificates" / "empirical_laws", Path.cwd().parent / "certificates" / "empirical_laws"):
    if (path / "notebook_setup.py").exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

import notebook_setup
from lyapunov import has_lyapunov
from utils import dask_parallel_map


METHODS = ["EF21", "EF"]
N_WORKER_SWEEP_CONFIGS = [
    {
        "n_workers": 2,
        "epsilons": np.linspace(0.05, 0.95, 10),
        "L_values": [1.0, 100, 1000.0],
        "kappa_values": [2, 100, 1000],
    },
    {
        "n_workers": 3,
        "epsilons": np.linspace(0.05, 0.95, 10),
        "L_values": [1.0, 1000.0],
        "kappa_values": [2, 100, 1000],
    },
    {
        "n_workers": 4,
        "epsilons": np.linspace(0.05, 0.95, 10),
        "L_values": [1.0, 1000.0],
        "kappa_values": [2, 1000],
    },
]

ETA_GRID_RESOLUTION = 40
TEST_IMPROVEMENT = 1 - 1e-2
BISECTION_TOL = 1e-7
MOSEK_SOLVE_KWARGS = notebook_setup.MOSEK_STRICT_SOLVE_KWARGS
SDPA_SOLVE_KWARGS = dict(notebook_setup.SDPA_HIGH_PRECISION_SOLVE_KWARGS)
DASK_SCHEDULER = "processes"
DASK_NUM_WORKERS = 14
def is_improvement(rho_ref, rho_new):
    return rho_ref > 0.0 and rho_new <= rho_ref * TEST_IMPROVEMENT


def reference_certificate(*, eps, n_workers, method, scale_info, solver, solve_kwargs, source_label):
    Ls = scale_info["Ls"]
    mus = scale_info["mus"]
    eta_theory = notebook_setup.empirical_eta_star(eps, Ls, mus, n_workers)
    lyap_kwargs = {
        "delta": 1 - eps,
        "n_workers": n_workers,
        "mus": mus,
        "Ls": Ls,
        "method": method,
        "use_simplified_lyapunov": False,
        "homogenous": method == "EF",
    }
    if n_workers == 2:
        # For n=2, use the Empirical Law 4.3 cubic candidate as the reference rate for both EF21 and EF.
        rho_star = notebook_setup.rho_opt_n2(eps, Ls, mus)
        if not np.isfinite(rho_star):
            return None, {"reason": "n2 cubic rate is not finite"}
        star_info = notebook_setup.warning_aware_solve(
            has_lyapunov,
            float(rho_star),
            eta=float(eta_theory),
            solver=solver,
            solve_kwargs=solve_kwargs,
            **lyap_kwargs,
        )
        if not star_info["ok_clean"]:
            return None, {
                "reason": f"{source_label} n2 cubic reference not cleanly certified (raw_ok={star_info['ok_raw']})",
                "warn_patterns": star_info["warn_patterns"],
            }
        return {
            "rho_star": float(rho_star),
            "eta_theory": float(eta_theory),
            "lyap_kwargs": lyap_kwargs,
            "solve_kwargs": solve_kwargs,
            "source": f"{source_label}_n2_cubic",
            "solver": solver,
            "scale_info": scale_info,
        }, None

    rho_star, _, _, warned_midpoints = notebook_setup.warning_aware_bisection(
        0.0,
        1.0,
        BISECTION_TOL,
        has_lyapunov,
        eta=float(eta_theory),
        solver=solver,
        solve_kwargs=solve_kwargs,
        **lyap_kwargs,
    )
    if rho_star is None:
        return None, {
            "reason": f"{source_label} bisection could not certify rho_star<=1 (warned_midpoints={warned_midpoints})",
        }
    return {
        "rho_star": float(rho_star),
        "eta_theory": float(eta_theory),
        "lyap_kwargs": lyap_kwargs,
        "solve_kwargs": solve_kwargs,
        "source": f"{source_label}_bisection",
        "solver": solver,
        "scale_info": scale_info,
    }, None


def verify_config(args):
    L_tuple, kappa_tuple, eps, n_workers, method = args
    scale_info = notebook_setup.scaled_problem_data_for_case(L_tuple, kappa_tuple)
    cert, failure = reference_certificate(
        eps=eps,
        n_workers=n_workers,
        method=method,
        scale_info=scale_info,
        solver="MOSEK",
        solve_kwargs=dict(MOSEK_SOLVE_KWARGS),
        source_label="mosek",
    )
    if cert is None:
        reason = failure.get("reason", "unknown failure") if failure else "unknown failure"
        warnings = failure.get("warn_patterns") if failure else None
        warning_text = f" warnings={warnings}" if warnings else ""
        return (
            f"FAIL: method={method} L={L_tuple} kappa={kappa_tuple} Eps={eps:.6f} | "
            f"could not certify reference rate with {notebook_setup.scale_label(scale_info)}: "
            f"{reason}{warning_text}"
        )

    rho_star = cert["rho_star"]
    rho_imp = rho_star * TEST_IMPROVEMENT
    mus = cert["lyap_kwargs"]["mus"]
    Ls = cert["lyap_kwargs"]["Ls"]
    eta_max = 2.0 * n_workers / np.sum(Ls + mus)
    mu_bar = float(np.mean(mus))
    L_bar = float(np.mean(Ls))
    if mu_bar <= 0.0 or L_bar <= 0.0:
        eta_lo, eta_hi = 0.0, float(eta_max)
    else:
        q = np.sqrt(np.clip(float(rho_star), 0.0, 1.0))
        eta_lo = max(0.0, (1.0 - q) / mu_bar)
        eta_hi = min(float(eta_max), (1.0 + q) / L_bar)
    if eta_lo > eta_hi:
        return None
    eta_grid = np.linspace(eta_lo, eta_hi, ETA_GRID_RESOLUTION)

    for eta_val in eta_grid:
        imp_result = notebook_setup.warning_aware_solve(
            has_lyapunov,
            rho_imp,
            eta=float(eta_val),
            solver=cert["solver"],
            solve_kwargs=cert["solve_kwargs"],
            **cert["lyap_kwargs"],
        )
        if not imp_result["ok_clean"]:
            continue

        rho_eta, _, _, _ = notebook_setup.warning_aware_bisection(
            0.0,
            1.0,
            BISECTION_TOL,
            has_lyapunov,
            eta=float(eta_val),
            solver=cert["solver"],
            solve_kwargs=cert["solve_kwargs"],
            **cert["lyap_kwargs"],
        )
        if rho_eta is None or not is_improvement(rho_star, rho_eta):
            continue

        sdpa_imp_result = notebook_setup.warning_aware_solve(
            has_lyapunov,
            rho_imp,
            eta=float(eta_val),
            solver="SDPA",
            solve_kwargs=SDPA_SOLVE_KWARGS,
            **cert["lyap_kwargs"],
        )
        if not sdpa_imp_result["ok_clean"]:
            continue

        scale_info = cert["scale_info"]
        return (
            f"FAIL: method={method} L={L_tuple} kappa={kappa_tuple} Eps={eps:.6f} | "
            f"rho_ref={rho_star:.10f} "
            f"rho_imp={rho_imp:.10f} "
            f"eta_theory={cert['eta_theory']:.8f} eta_candidate={float(eta_val):.8f} "
            f"reference_source={cert['source']} "
            f"scaling={notebook_setup.scale_label(scale_info)} "
            f"sdpa_confirmation=spot_check_mosek_threshold"
        )

    return None


def run_rigorous_check(sweep_config, method="EF21"):
    n_workers = int(sweep_config["n_workers"])
    worker_configs = notebook_setup.worker_L_kappa_configs(
        list(sweep_config["L_values"]),
        list(sweep_config["kappa_values"]),
        n_workers,
        dedup_permutations=True,
    )
    configs = [
        (L_cfg, kappa_cfg, float(eps), n_workers, method)
        for (L_cfg, kappa_cfg) in worker_configs
        for eps in np.array(sweep_config["epsilons"], dtype=float)
    ]
    print(f"--- Verification Suite: Empirical Law 4.1 (Step Size), n={n_workers}, method={method} ---")
    print(f"Checking {len(configs)} configurations with {DASK_NUM_WORKERS} workers...")
    print("Solver path: MOSEK screen, SDPA confirmation")
    if not configs:
        print("No configurations in this sweep.\n")
        return []

    results = dask_parallel_map(
        verify_config,
        configs,
        scheduler=DASK_SCHEDULER,
        show_progress=True,
        num_workers=DASK_NUM_WORKERS,
    )
    failures = [msg for msg in results if msg]
    if not failures:
        print("ALL CHECKS PASSED\n")
        return []

    for msg in failures:
        print(msg)
    print(f"FAILURES={len(failures)}\n")
    return failures


if __name__ == "__main__":
    total_failures = []
    for sweep_config in N_WORKER_SWEEP_CONFIGS:
        for method in METHODS:
            failures = run_rigorous_check(sweep_config=sweep_config, method=method)
            total_failures.extend(failures)

    if total_failures:
        print(f"TOTAL_FAILURES={len(total_failures)}")
        raise SystemExit(1)
    print("ALL STEP-SIZE CHECKS PASSED")

--- Verification Suite: Empirical Law 4.1 (Step Size), n=2, method=EF21 ---
Checking 450 configurations with 14 workers...
Solver path: MOSEK screen, SDPA confirmation


compute: 100%|██████████| 450/450 [00:29<00:00, 15.44it/s]


ALL CHECKS PASSED

--- Verification Suite: Empirical Law 4.1 (Step Size), n=2, method=EF ---
Checking 450 configurations with 14 workers...
Solver path: MOSEK screen, SDPA confirmation


compute: 100%|██████████| 450/450 [00:36<00:00, 12.25it/s]


ALL CHECKS PASSED

--- Verification Suite: Empirical Law 4.1 (Step Size), n=3, method=EF21 ---
Checking 560 configurations with 14 workers...
Solver path: MOSEK screen, SDPA confirmation


...

ALL CHECKS PASSED

--- Verification Suite: Empirical Law 4.1 (Step Size), n=4, method=EF21 ---
Checking 350 configurations with 14 workers...
Solver path: MOSEK screen, SDPA confirmation


compute: 100%|██████████| 350/350 [01:39<00:00,  3.53it/s]


ALL CHECKS PASSED

--- Verification Suite: Empirical Law 4.1 (Step Size), n=4, method=EF ---
Checking 350 configurations with 14 workers...
Solver path: MOSEK screen, SDPA confirmation


compute: 100%|██████████| 350/350 [25:27<00:00,  4.36s/it]

ALL CHECKS PASSED

ALL STEP-SIZE CHECKS PASSED
